https://www.youtube.com/watch?v=elRHaDiNkY8&list=PL-HzNlEIBQqKQ2CeJxgNDTb7xMBspa2ue&index=10

In [10]:
!pip install alpaca-py pandas numpy plotly backtesting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 4.8 MB/s eta 0:00:00


In [52]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backtesting import Backtest, Strategy
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

# ==========================================
# Configuración de API de Alpaca
# ==========================================
API_KEY = "PKILMNVKETV5POH5GOPM37W37E"
SECRET_KEY = "DcQGFZBAdWpLc7iuBUVdAn8PdAYrnpPF93CCaRaXsjcG"

cliente_datos = StockHistoricalDataClient(API_KEY, SECRET_KEY)
print("Librerías importadas y cliente de Alpaca inicializado.")

Librerías importadas y cliente de Alpaca inicializado.


In [70]:
symbol = 'MSFT'

# ==========================================
# Descarga de datos (Velas de 1 Hora)
# ==========================================
# Descargamos los últimos 60 días, terminando ayer
fecha_fin = datetime.now() - timedelta(days=1)
fecha_inicio = fecha_fin - timedelta(days=360)

parametros_peticion = StockBarsRequest(
    symbol_or_symbols=symbol,
    timeframe=TimeFrame.Hour,
    start=fecha_inicio,
    end=fecha_fin,
    feed="iex" # Feed gratuito
)

print("Descargando datos históricos de Alpaca...")
barras = cliente_datos.get_stock_bars(parametros_peticion)

# Extraer y limpiar el DataFrame
df_1h = barras.df.loc[symbol].copy()
df_1h.rename(columns={
    'open': 'Open', 'high': 'High', 'low': 'Low',
    'close': 'Close', 'volume': 'Volume'
}, inplace=True)

print(f"¡Datos descargados con éxito! Total de velas: {len(df_1h)}")

Descargando datos históricos de Alpaca...
¡Datos descargados con éxito! Total de velas: 1710


In [71]:
# ==========================================
# Función Generadora de Señales
# ==========================================
def calcular_señales_breakout(df, ventana_volumen, multiplicador_volumen):
    df_est = df.copy()
    df_est['Vol_Promedio'] = df_est['Volume'].rolling(window=ventana_volumen).mean()

    nivel_alto = np.nan
    nivel_bajo = np.nan
    buscando_ruptura = False

    señales, sl_niveles, tp_niveles = [], [], []

    for index, row in df_est.iterrows():
        señal, sl, tp = 0, np.nan, np.nan

        # A. Evaluar si hay ruptura
        if buscando_ruptura:
            if row['Close'] > nivel_alto:
                señal = 1  # Long
                riesgo = row['Close'] - nivel_bajo
                sl = nivel_bajo
                tp = row['Close'] + (2 * riesgo) # Beneficio 2R
                buscando_ruptura = False

            elif row['Close'] < nivel_bajo:
                señal = -1 # Short
                riesgo = nivel_alto - row['Close']
                sl = nivel_alto
                tp = row['Close'] - (2 * riesgo) # Beneficio 2R
                buscando_ruptura = False

        # B. Detectar nuevo pico de volumen
        if pd.notna(row['Vol_Promedio']):
            if row['Volume'] > (row['Vol_Promedio'] * multiplicador_volumen):
                nivel_alto = row['High']
                nivel_bajo = row['Low']
                buscando_ruptura = True

        señales.append(señal)
        sl_niveles.append(sl)
        tp_niveles.append(tp)

    df_est['Señal'] = señales
    df_est['SL'] = sl_niveles
    df_est['TP'] = tp_niveles

    return df_est

# Aplicamos la función a nuestros datos
df_con_señales = calcular_señales_breakout(df_1h, ventana_volumen=50, multiplicador_volumen=2)
print("Señales, Stop Loss y Take Profit calculados.")
df_con_señales.Señal.value_counts(normalize=True)*100

Señales, Stop Loss y Take Profit calculados.


,proportion
Señal,
0,89.064327
1,6.140351
-1,4.795322


In [72]:
# ==========================================
# Visualización con Plotly
# ==========================================
def graficar_estrategia_plotly_contiguo(df):
    fechas_texto = df.index.strftime('%Y-%m-%d %H:%M')

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03,
        row_heights=[0.7, 0.3], subplot_titles=('Precio y Señales', 'Volumen')
    )

    # Velas
    fig.add_trace(go.Candlestick(x=fechas_texto, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'], name='Precio'), row=1, col=1)

    # Compras
    compras = df[df['Señal'] == 1]
    if not compras.empty:
        fig.add_trace(go.Scatter(x=compras.index.strftime('%Y-%m-%d %H:%M'), y=compras['Low'] - (compras['Close'] * 0.002), mode='markers', marker=dict(symbol='triangle-up', color='blue', size=14), name='Long'), row=1, col=1)

    # Ventas
    ventas = df[df['Señal'] == -1]
    if not ventas.empty:
        fig.add_trace(go.Scatter(x=ventas.index.strftime('%Y-%m-%d %H:%M'), y=ventas['High'] + (ventas['Close'] * 0.002), mode='markers', marker=dict(symbol='triangle-down', color='orange', size=14), name='Short'), row=1, col=1)

    # Volumen y Media
    fig.add_trace(go.Bar(x=fechas_texto, y=df['Volume'], name='Volumen', marker_color='rgba(158,202,225,0.8)'), row=2, col=1)
    if 'Vol_Promedio' in df.columns:
        fig.add_trace(go.Scatter(x=fechas_texto, y=df['Vol_Promedio'], mode='lines', line=dict(color='purple', width=2), name='Media Vol'), row=2, col=1)

    fig.update_layout(title='Breakout por Volumen', xaxis_rangeslider_visible=False, template='plotly_dark', height=700)
    fig.update_xaxes(type='category', nticks=10, tickangle=45)
    fig.show()

# Mostrar el gráfico interactivo
graficar_estrategia_plotly_contiguo(df_con_señales)

In [73]:
# ==========================================
# Backtesting
# ==========================================
class EjecutorDeSeñales(Strategy):
    def init(self):
        pass

    def next(self):
        # Si ya hay posición abierta, esperamos a que toque SL o TP
        if self.position:
            return

        señal_actual = self.data.Señal[-1]

        if señal_actual == 1:
            self.buy(sl=self.data.SL[-1], tp=self.data.TP[-1])
        elif señal_actual == -1:
            self.sell(sl=self.data.SL[-1], tp=self.data.TP[-1])

# Configurar el motor de backtesting
bt = Backtest(
    df_con_señales,
    EjecutorDeSeñales,
    cash=10000,           # Capital inicial: $10,000
    commission=0.002,     # Comisión: 0.2%
    exclusive_orders=True # No permite hedging (long y short a la vez)
)

# Ejecutar y mostrar métricas
resultados = bt.run()
print("--- ESTADÍSTICAS DEL BACKTEST ---")
print(resultados)

# Generar el gráfico interactivo del backtest (abre en nueva pestaña/ventana)
bt.plot()

Backtest.run:   0%|          | 0/1709 [00:00<?, ?bar/s]

/tmp/ipykernel_1019/2644171692.py:30: UserWarning:

Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.



--- ESTADÍSTICAS DEL BACKTEST ---
Start                     2025-03-11 13:00...
End                       2026-03-04 20:00...
Duration                    358 days 07:00:00
Exposure Time [%]                    61.81287
Equity Final [$]                   5780.69911
Equity Peak [$]                       10000.0
Commissions [$]                    1980.48909
Return [%]                          -42.19301
Buy & Hold Return [%]                50.50156
Return (Ann.) [%]                   -42.83079
Volatility (Ann.) [%]                11.07717
CAGR [%]                            -31.98701
Sharpe Ratio                         -3.86658
Sortino Ratio                        -2.63456
Calmar Ratio                         -0.94322
Alpha [%]                           -41.41667
Beta                                 -0.01537
Max. Drawdown [%]                   -45.40926
Avg. Drawdown [%]                   -45.40926
Max. Drawdown Duration      348 days 07:00:00
Avg. Drawdown Duration      348 days 07:00:00


/usr/local/lib/python3.12/dist-packages/bokeh/util/serialization.py:242: UserWarning:

no explicit representation of timezones available for np.datetime64



GridPlot(id='p5324', ...)